# Publish FT BERT

In [1]:
import os
from huggingface_hub import login
from huggingface_hub import HfApi
from sentence_transformers import SentenceTransformer, models

Conversion to sentence transformer

In [3]:
# Load your BERT model checkpoint
word_embedding_model = models.Transformer('/storage/Tax_Law_RAG/german-tax-law/output/EStG_Subtitle_Classification_BERT')

# Use CLS pooling explicitly
pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_cls_token=True,
    pooling_mode_mean_tokens=False,
    pooling_mode_max_tokens=False
)

model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

# Save as a Sentence Transformers model locally
model.save('/storage/Tax_Law_RAG/german-tax-law/output/EStG_Subtitle_Classification_BERT_st')

Push to hugging face

In [ ]:
login("")  # Token goes here!  os.getenv("HF_TOKEN")

api = HfApi(token="")
api.upload_folder(
    folder_path="/storage/Tax_Law_RAG/german-tax-law/output/EStG_Subtitle_Classification_BERT_st",
    repo_id="ninoid/sentence-transformers-EStG-bert",
    repo_type="model",
)

CommitInfo(commit_url='https://huggingface.co/chandlerNick/sentence-transformers-usc26-bert/commit/4b7036333d5b5d9294b11da36054ea29e7be7a79', commit_message='Upload folder using huggingface_hub', commit_description='', oid='4b7036333d5b5d9294b11da36054ea29e7be7a79', pr_url=None, repo_url=RepoUrl('https://huggingface.co/chandlerNick/sentence-transformers-usc26-bert', endpoint='https://huggingface.co', repo_type='model', repo_id='chandlerNick/sentence-transformers-usc26-bert'), pr_revision=None, pr_num=None)

Test that langchain can use it

In [2]:
from langchain.embeddings import HuggingFaceEmbeddings
import torch
print(torch.cuda.is_available())  
print(torch.cuda.get_device_name(0))  


True
Tesla V100S-PCIE-32GB


# not using model from huggingface because of weird errors
device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = HuggingFaceEmbeddings(
    model_name="ninoid/sentence-transformers-EStG-bert",
    model_kwargs={"device": device},
    encode_kwargs={"show_progress_bar": False}
)


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = HuggingFaceEmbeddings(
    model_name="/storage/Tax_Law_RAG/german-tax-law/output/EStG_Subtitle_Classification_BERT_st",
    model_kwargs={"device": device}
)


In [6]:
text = "Gewinn ist der Unterschiedsbetrag zwischen dem Betriebsvermögen am Schluss des Wirtschaftsjahres und dem Betriebsvermögen am Schluss des vorangegangenen Wirtschaftsjahres."
embedding = embedding_model.embed_query(text)

print("Embedding size:", len(embedding))
print("First 5 values:", embedding[:5])

Embedding size: 768
First 5 values: [-0.030769817531108856, -0.6778433322906494, 0.7218289375305176, 0.223257377743721, -0.9641093611717224]
